# Homework 2 — Question 2

`%pip install numpy pandas torch scikit-learn`


## (a) Cardinality and data splitting

The dataset contains 397,924 transactions and no missing values. `StockCode` has **3,665** unique values and `CustomerID` has **4,339** unique values. I treat both features as categorical, including the numeric-looking customer IDs.

One-hot encoding would create 8,004 columns. A dense float32 matrix for the entire dataset would require approximately 12.7 GB, excluding overhead. Sparse one-hot encoding is feasible, but target encoding and embeddings provide much smaller representations.

I split the data randomly into **70% training, 15% validation, and 15% test**, using seed 42. The sets contain 278,546, 59,689, and 59,689 observations, respectively. The same split is used for all three models.

`Quantity` is highly skewed: the median is **6**, while the maximum is **80,995**. I therefore predict $t=\log(1+\mathrm{Quantity})$ and minimize mean squared error on this scale:

$$L=\frac1n\sum_{i=1}^n(t_i-\hat t_i)^2.$$

I report test log-MSE as the main metric and MAE in the original quantity units as a second metric. For MAE, I transform predictions back using $\hat y=\max\{\exp(\hat t)-1,0\}$. Lower values of both metrics are better. No transactions are removed.


### Imports

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

np.random.seed(42)
torch.manual_seed(42)
torch.set_num_threads(2)



### Explore and split the data

In [2]:
data = pd.read_csv("retail.csv")
x = data[["StockCode", "CustomerID"]].astype(str)
y = data["Quantity"].to_numpy(dtype="float32")

print("Data shape:", data.shape)
print("Unique categories:\n", x.nunique().to_string())
print("Missing values:", int(data.isna().sum().sum()))
print("Quantity median:", float(np.median(y)))
print("Quantity maximum:", float(np.max(y)))

x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.30, random_state=42
)
x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.50, random_state=42
)
print("Train / validation / test:", len(y_train), len(y_val), len(y_test))

# All models predict log(1 + Quantity).
t_train = np.log1p(y_train)
t_val = np.log1p(y_val)
t_test = np.log1p(y_test)



Data shape: (397924, 3)
Unique categories:
 StockCode     3665
CustomerID    4339
Missing values: 0
Quantity median: 6.0
Quantity maximum: 80995.0
Train / validation / test: 278546 59689 59689


## (b) Target encoding

I encode each feature using a smoothed mean of the original `Quantity` for that category. I use scikit-learn's `TargetEncoder` with smoothing strength 20 and five-fold cross-fitting. Training rows receive out-of-fold encodings, so their own target values are not used to construct their features. Validation and test encodings use training labels only; unseen categories receive the training mean.

I apply `log1p` to the two positive encoded features, then standardize them using training-set statistics. These steps reduce feature skew and put the inputs on comparable scales. A neural network with two inputs and hidden layers of 64 and 32 ReLU neurons predicts log quantity.

All three models use Adam with learning rate 0.001, batch size 1,024, and at most 50 epochs. I stop after five consecutive epochs without improvement in validation log-MSE and restore the best parameters. I use fixed settings and do not tune on the test set.

The target encoding model has **test log-MSE 0.401368** and **test MAE 6.930372**.


### Target encoding

In [3]:
# This encoder comes from scikit-learn (not category_encoders).
# fit_transform uses out-of-fold encoding for the training rows.
# Validation/test rows are encoded using only training labels.
encoder = TargetEncoder(
    target_type="continuous", smooth=20.0, cv=5,
    shuffle=True, random_state=42
)
a_train = encoder.fit_transform(x_train, y_train)
a_val = encoder.transform(x_val)
a_test = encoder.transform(x_test)

# The encoding uses means of original Quantity, as requested.
# Log-transform these positive features to reduce their skew, then scale.
scaler = StandardScaler()
a_train = scaler.fit_transform(np.log1p(a_train)).astype("float32")
a_val = scaler.transform(np.log1p(a_val)).astype("float32")
a_test = scaler.transform(np.log1p(a_test)).astype("float32")



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


## (c) Embeddings

I use a separate embedding layer for `StockCode` and `CustomerID`, with **16 dimensions per feature**. I concatenate the two vectors into a 32-dimensional input, followed by the same 64- and 32-neuron hidden layers. The embeddings and network weights are learned together by minimizing log-scale MSE. Each embedding layer reserves ID 0, a fixed zero vector, for categories not observed during training.

The embedding model has **test log-MSE 0.292724** and **test MAE 6.245780**.

The embedding model has lower test log-MSE than the target encoding model.


### Category IDs for embeddings

In [4]:
# IDs and embedding sizes are based only on the training categories.
# ID 0 is reserved for unseen categories.
b_train = np.zeros((len(x_train), 2), dtype="int64")
b_val = np.zeros((len(x_val), 2), dtype="int64")
b_test = np.zeros((len(x_test), 2), dtype="int64")
sizes = []

for j, col in enumerate(x.columns):
    values = x_train[col].unique()
    ids = {value: i + 1 for i, value in enumerate(values)}
    sizes.append(len(values) + 1)
    b_train[:, j] = x_train[col].map(ids).fillna(0).to_numpy()
    b_val[:, j] = x_val[col].map(ids).fillna(0).to_numpy()
    b_test[:, j] = x_test[col].map(ids).fillna(0).to_numpy()

print("Training categories: StockCode =", sizes[0] - 1,
      "; CustomerID =", sizes[1] - 1)



Training categories: StockCode = 3595 ; CustomerID = 4311


## (d) Target encoding and embeddings together

I concatenate the two target-encoded inputs with the two 16-dimensional embeddings. This gives **34 inputs** to the same neural network. The loss, data split, hidden-layer sizes, and training settings are unchanged.

The combined model has **test log-MSE 0.297335** and **test MAE 6.242244**.

Its test log-MSE is lower than Target and higher than Embedding.


### Three models

In [5]:
# Each model has the same hidden layers, loss and training settings.
class Model(nn.Module):
    def __init__(self, kind):
        super().__init__()
        self.kind = kind

        if kind == "Target":
            size = 2
        else:
            self.stock = nn.Embedding(sizes[0], 16, padding_idx=0)
            self.customer = nn.Embedding(sizes[1], 16, padding_idx=0)
            size = 32 if kind == "Embedding" else 34

        self.net = nn.Sequential(
            nn.Linear(size, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, a, b):
        if self.kind == "Target":
            h = a
        else:
            h = torch.cat([self.stock(b[:, 0]), self.customer(b[:, 1])], dim=1)
            if self.kind == "Both":
                h = torch.cat([a, h], dim=1)
        return self.net(h).squeeze(1)




### Training function

In [6]:
train = TensorDataset(
    torch.tensor(a_train), torch.tensor(b_train), torch.tensor(t_train)
)
a_val_t = torch.tensor(a_val)
b_val_t = torch.tensor(b_val)
t_val_t = torch.tensor(t_val)
loss_fn = nn.MSELoss()


def fit(kind):
    torch.manual_seed(42)
    model = Model(kind)
    loader = DataLoader(train, batch_size=1024, shuffle=True)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    best_loss = float("inf")
    best_epoch = 0
    wait = 0
    history = []

    for epoch in range(50):
        model.train()
        total = 0.0

        for a, b, t in loader:
            optimizer.zero_grad()
            pred = model(a, b)
            loss = loss_fn(pred, t)
            loss.backward()
            optimizer.step()
            total += loss.item() * len(t)

        model.eval()
        with torch.no_grad():
            pred = model(a_val_t, b_val_t)
            val_loss = loss_fn(pred, t_val_t).item()

        history.append([total / len(train), val_loss])
        if val_loss < best_loss:
            best_loss = val_loss
            best_epoch = epoch + 1
            best_model = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1

        if (epoch + 1) % 10 == 0:
            print(f"{kind}: epoch {epoch + 1}, validation log-MSE = {val_loss:.6f}")
        if wait >= 5:
            break

    model.load_state_dict(best_model)
    model.eval()
    pd.DataFrame(history, columns=["train_log_mse", "val_log_mse"]).to_csv(
        kind.lower() + "_history.csv", index=False
    )
    print(f"{kind}: best epoch = {best_epoch}, validation log-MSE = {best_loss:.6f}")
    return model, best_epoch, best_loss




### Fit all models (no test-set model selection)

In [7]:
models = {}
info = {}
for kind in ["Target", "Embedding", "Both"]:
    models[kind], epoch, loss = fit(kind)
    info[kind] = [epoch, loss]



Target: epoch 10, validation log-MSE = 0.401342
Target: epoch 20, validation log-MSE = 0.400526
Target: best epoch = 22, validation log-MSE = 0.399635
Embedding: epoch 10, validation log-MSE = 0.314808
Embedding: epoch 20, validation log-MSE = 0.296863
Embedding: epoch 30, validation log-MSE = 0.293527
Embedding: epoch 40, validation log-MSE = 0.291838
Embedding: best epoch = 40, validation log-MSE = 0.291838
Both: epoch 10, validation log-MSE = 0.318593
Both: epoch 20, validation log-MSE = 0.297742
Both: epoch 30, validation log-MSE = 0.300090
Both: best epoch = 33, validation log-MSE = 0.294837


### Test-set evaluation

In [8]:
rows = []
for kind in ["Target", "Embedding", "Both"]:
    with torch.no_grad():
        pred = models[kind](torch.tensor(a_test), torch.tensor(b_test)).numpy()
    quantity = np.maximum(np.expm1(pred), 0)
    rows.append({
        "Model": kind,
        "Best epoch": info[kind][0],
        "Validation log-MSE": info[kind][1],
        "Test log-MSE": mean_squared_error(t_test, pred),
        "Test MAE": mean_absolute_error(y_test, quantity)
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
results.to_csv("results.csv", index=False)



    Model  Best epoch  Validation log-MSE  Test log-MSE  Test MAE
   Target          22            0.399635      0.401391  6.927233
Embedding          40            0.291838      0.292773  6.232802
     Both          33            0.294837      0.297840  6.282616


### Simple constant baseline on the same test set

In [9]:
pred = np.full(len(t_test), t_train.mean())
print("Constant baseline test log-MSE:", mean_squared_error(t_test, pred))


Constant baseline test log-MSE: 0.9966404438018799


## (e) Interpretation and test results

| Model | Best epoch | Validation log-MSE | Test log-MSE | Test MAE |
|---|---:|---:|---:|---:|
| Target | 22 | 0.399615 | 0.401368 | 6.930372 |
| Embedding | 40 | 0.292367 | 0.292724 | 6.245780 |
| Both | 33 | 0.294854 | 0.297335 | 6.242244 |

The **Embedding** model achieves the lowest test log-MSE in this experiment.

Target encoding summarizes each category with one average quantity. This is compact but loses information about other differences between products or customers. Embeddings learn several numerical features for each category. The network can combine product and customer vectors to capture patterns that a single average cannot represent.

The combined model supplies both an explicit average and learned vectors. This can help learning, but it is not guaranteed to improve performance: the information may overlap, and additional inputs can also increase overfitting. These are possible explanations, rather than evidence that a particular embedding dimension has a known meaning.

The comparison uses the same held-out test set and loss definition for all models. Log-MSE emphasizes errors on the transformed scale, while original-scale MAE measures a different aspect of prediction quality. These results apply to a random split of transactions, where many customers and products occur in both training and test sets.
